In [1]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.6.0+cu124
True
NVIDIA GeForce RTX 2050


In [2]:
import transformers
import datasets

print(transformers.__version__)
print(datasets.__version__)

c:\Users\arote\anaconda3\envs\animal_fixed\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.52.4
2.18.0


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from datasets import Dataset

import torch

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

print(torch.cuda.is_available())

True


In [4]:
df = pd.read_parquet(
    "../../data/processed/classification_dataset_final.parquet"
)

print(df.shape)

df.head()

(26688, 4)


,case_name,year,legal_summary,category
0,A_K_Gopalan_vs_The_State_Of_Madras_Union_Of_In...,1950,The petitioner who was detained under the Pre...,Constitutional
1,A_M_Mair_Co_vs_Gordhandass_Sagarmull_on_30_Nov...,1950,A.M. Mair & Co vs Gordhandass Sagarmull on 30...,Contract
2,Abdulla_Ahmed_vs_Animendra_Kissen_Mitter_on_14...,1950,"The appellant, an estate broker, was employed...",Property
3,Arjuna_Lal_Misra_vs_The_State_on_30_November_1...,1950,"Fazl Ali, J. 1. I do not wish to express diss...",Criminal
4,Ashutosh_Lahiry_vs_The_State_Of_Delhi_And_Anr_...,1950,"""You came to Delhi on March 27, 1950 and held...",Civil


In [5]:

label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["category"]
)

print(label_encoder.classes_)

['Civil' 'Company_Commercial' 'Constitutional' 'Contract' 'Criminal'
 'Family' 'Property' 'Service']


In [6]:

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print(train_df.shape)
print(test_df.shape)

(21350, 5)
(5338, 5)


In [7]:
train_dataset = Dataset.from_pandas(
    train_df[
        ["legal_summary","label"]
    ]
)

test_dataset = Dataset.from_pandas(
    test_df[
        ["legal_summary","label"]
    ]
)

print(train_dataset)

Dataset({
    features: ['legal_summary', 'label', '__index_level_0__'],
    num_rows: 21350
})


In [8]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [9]:
def tokenize(batch):

    return tokenizer(
        batch["legal_summary"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

Map: 100%|██████████| 5338/5338 [00:02<00:00, 2169.95 examples/s]


In [10]:
train_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

test_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

In [11]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=8
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
training_args = TrainingArguments(
    output_dir="../../models/classification/distilbert",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    load_best_model_at_end=True,

    fp16=True
)

In [13]:
import evaluate

accuracy_metric = evaluate.load(
    "accuracy"
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

In [14]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [15]:
import torch

print(torch.cuda.get_device_name(0))

print(
    round(
        torch.cuda.get_device_properties(0).total_memory
        /1024**3,
        2
    ),
    "GB"
)

NVIDIA GeForce RTX 2050
4.0 GB


In [16]:
import numpy as np

print(np.__version__)

1.26.4


In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.621400,0.671415,0.808355
2,0.583600,0.685348,0.811727
3,0.536700,0.742844,0.813226


TrainOutput(global_step=16014, training_loss=0.6305393610194752, metrics={'train_runtime': 2323.0641, 'train_samples_per_second': 27.571, 'train_steps_per_second': 6.893, 'total_flos': 4242722370969600.0, 'train_loss': 0.6305393610194752, 'epoch': 3.0})

In [18]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.6714152097702026, 'eval_accuracy': 0.8083551892094417, 'eval_runtime': 35.4691, 'eval_samples_per_second': 150.497, 'eval_steps_per_second': 37.638, 'epoch': 3.0}


In [19]:
trainer.save_model(
    "../../models/classification/distilbert_final"
)

tokenizer.save_pretrained(
    "../../models/classification/distilbert_final"
)

print("Model Saved")

Model Saved
